# EXP-014 — Sửa lệch lớp kiểu inductive (ước lượng từ ảnh train không nhãn)
EXP-013: trừ trung bình điểm theo lớp **trên tập test** (`center-test`, transductive) đưa CLIP gốc 42.04 → 50.43 Mean4 UAR. Ở đây độ lệch được ước lượng từ **ảnh mặt không nhãn ngoài tập test** → zero-shot inductive.

- Điểm = cos(ảnh, prompt) ⇒ trung bình điểm lớp c trên tập tham chiếu = μ·t_c, với μ = trung bình feature ảnh đã chuẩn hoá (sau head). Điểm đã sửa: S − α·b.
- Tham chiếu: **CAER-S train** (chính, khác miền với mọi tập test), AffectNet-train 49k (phụ — cùng miền với AffectNet7/8 test). Feature CLIP đã cache (tiền xử lý `raw` = P0).
- Chỉ test, GPU nào cũng được.

**Tiêu chí (chọn trước; Ensemble Five, P0, CAER-S toàn bộ, α = 1):**
- Chính: CLIP gốc + center-CAER ≥ 47.56 Mean4 UAR (= EXP-010 e5 ở `none`).
- Phụ 1: |center-CAER − center-test| ≤ 1.0 trên CLIP gốc và EXP-010 e5.
- Phụ 2: EXP-010 e5 + center-CAER − CLIP gốc + center-CAER ≥ +1.0.

# Cấu hình

In [ ]:
SEEDS = [1, 2, 3]
BRANCH = "exp/012-qwen-teacher"   # chỉ cần models/clip, models/Text.py, data_loader
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
ROOT = "/content/drive/MyDrive/ExpCLIP_output"
OUT_DIR = f"{ROOT}/exp014"
EXP004_DIR = f"{ROOT}/exp004"
EXP010_DIR = f"{ROOT}/exp010"
REF_FEATS = {"CAER": f"{ROOT}/exp001/full/clip_feats_l14.npz",            # CAER-S train, raw (EXP-003)
             "AffNet": f"{ROOT}/exp010/pre_affectnet/clip_feats_l14.npz"}  # AffectNet-train 49k, raw (EXP-010)
ALPHAS = [0.5, 1.0, 1.5]
SUBSETS = [("all", 0), (5000, 0), (5000, 1), (5000, 2), (500, 0), (500, 1), (500, 2)]   # (số ảnh, lần chọn)

import os, glob

# Mount Drive, clone repo, giải nén test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!pip install -q ftfy "setuptools<81"

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB
Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 486, done.
remote: Counting objects: 100% (486/486), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 486 (delta 352), reused 410 (delta 277), pack-reused 0 (from 0)
Receiving objects: 100% (486/486), 32.38 MiB | 33.03 MiB/s, done.
Resolving deltas: 100% (352/352), done.
/content/Exp-CLIP
Already on 'exp/012-qwen-teacher'
Your branch is up to date with 'origin/exp/012-qwen-teacher'.
Already up to date.
558dba2 (HEAD -> exp/012-qwen-teacher, origin/exp/012-qwen-teacher) EXP-012: Qwen2.5-VL teacher targets + head output dim from target


# Checkpoint

In [ ]:
def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

FAMILIES = {   # họ head → {seed: đường dẫn}
    "EXP-000": {s: find_ckpt(ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004 e30": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
    "EXP-010 e5": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e5-model.pth" for s in SEEDS},
    "EXP-010 e10": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e10-model.pth" for s in SEEDS},
}
for f, d in FAMILIES.items():
    for s, p in list(d.items()):
        if p is None or not os.path.exists(p):
            print(f"CẢNH BÁO: {f} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
    print(f, "→", len(d), "seed")
PAPER = "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"

EXP-000 → 3 seed
EXP-004 e30 → 3 seed
EXP-010 e5 → 3 seed
EXP-010 e10 → 3 seed


# Feature CLIP của ảnh test (P0 × gốc/lật) và prompt

In [ ]:
import sys, re, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
CFG_NAMES = ["P0", "P0+flip"]
BALANCES = ["none", "center", "sinkhorn"]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[ds] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", image_transform("raw"))
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        f0, f1 = [], []
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
        img[ds_name] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name][0].shape), classes[ds_name], np.bincount(y[ds_name]))

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 890M/890M [00:13<00:00, 71.5MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 

RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear'] [1185  478  680  162  329  160   74]
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear'] [500 500 500 500 500 500 500]
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt'] [500 500 500 500 500 500 500 500]
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt'] [ 929  449 1274  322  450   21   98   30]


# Feature ảnh tham chiếu (không nhãn)

In [15]:
ref = {}
for name, path in REF_FEATS.items():
    if not os.path.exists(path):
        print(f"CẢNH BÁO: không thấy {path} → bỏ tham chiếu {name}"); continue
    ref[name] = torch.from_numpy(np.load(path)["feats"].astype(np.float32))
    print(name, tuple(ref[name].shape))
assert "CAER" in ref, "thiếu tham chiếu chính CAER-S"

CAER (48938, 768)
AffNet (48887, 768)


# Điểm lớp và độ lệch theo lớp
- `b_test` = trung bình điểm lớp trên chính tập test (transductive, = `center` của EXP-013).
- `b_ref` = μ·t_c, μ = trung bình feature ảnh tham chiếu (sau head, đã chuẩn hoá); trung bình qua prompt của lớp như điểm.
- Head `ens` (nhiều W): sửa từng head bằng độ lệch của chính nó rồi trung bình.

In [16]:
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def load_W(path):
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def proj(x, W):
    x = x if W is None else x @ W.T
    return F.normalize(x, dim=-1)

def text_feats(W, ds_name, pn):
    return proj(txt[(ds_name, pn)], W)

def class_scores(W, ds_name, cfg, pn):
    f0, f1 = img[ds_name]
    i = (f0 + f1) / 2 if cfg.endswith("+flip") else f0   # TTA: trung bình trước head
    return (proj(i, W) @ text_feats(W, ds_name, pn).T).view(len(i), N_CLS[ds_name], -1).mean(-1).double().numpy()

MU = {}   # (id head, tên tham chiếu, n, lần chọn) → μ
def mu(W, wid, rname, n, rep):
    key = (wid, rname, n, rep)
    if key not in MU:
        x = ref[rname]
        if n != "all":
            x = x[torch.randperm(len(x), generator=torch.Generator().manual_seed(rep))[:n]]
        Wd = None if W is None else W.to(DEV)
        acc = torch.zeros(x.shape[1] if W is None else W.shape[0], device=DEV, dtype=torch.float64)
        for c in torch.split(x, 8192):
            acc += proj(c.to(DEV), Wd).double().sum(0)
        MU[key] = (acc / len(x)).float().cpu()
    return MU[key]

def ref_bias(W, wid, ds_name, pn, rname, n, rep):
    return (mu(W, wid, rname, n, rep) @ text_feats(W, ds_name, pn).T).view(N_CLS[ds_name], -1).mean(-1).double().numpy()

# cách sửa: (tên, tham chiếu, n, lần chọn, α)
METHODS = [("none", None, None, None, 0.0), ("center-test", "test", None, None, 1.0)]
METHODS += [(f"center-{r}", r, "all", 0, a) for r in ref for a in ALPHAS]
METHODS += [(f"center-CAER", "CAER", n, rep, 1.0) for n, rep in SUBSETS if n != "all"]

def corrected(Ws, ds_name, cfg, pn, method):
    _, rname, n, rep, a = method
    out = []
    for wid, W in Ws:
        S = class_scores(W, ds_name, cfg, pn)
        if rname == "test":
            S = S - a * S.mean(0, keepdims=True)
        elif rname is not None:
            S = S - a * ref_bias(W, wid, ds_name, pn, rname, n, rep)
        out.append(S)
    return np.mean(out, axis=0)

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    frac = np.bincount(p_, minlength=n_cls) / len(p_) * 100
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc, "frac": frac}

In [17]:
Wf = {f: {s: load_W(p) for s, p in d.items()} for f, d in FAMILIES.items() if f != "EXP-010 e10"}
MODELS = {"CLIP gốc": [(0, [("clip", None)])], "Paper": [(0, [("paper", load_W(PAPER))])]}   # tên → [(seed, [(id, W)])]
for f, d in Wf.items():
    MODELS[f] = [(s, [(f"{f}|{s}", W)]) for s, W in d.items()]
MODELS["ens chéo 004e30+010e5"] = [(0, [(f"{f}|{s}", W) for f in ["EXP-004 e30", "EXP-010 e5"] for s, W in Wf[f].items()])]

rows = []
for m, entries in MODELS.items():
    for s, Ws in entries:
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    for meth in METHODS:
                        if meth[2] not in (None, "all") and (pn != BASE_PROMPT or cfg != "P0"):
                            continue   # ablation kích thước tham chiếu: chỉ cấu hình chính
                        r = metrics(y[ds_name], corrected(Ws, ds_name, cfg, pn, meth).argmax(1), N_CLS[ds_name])
                        rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn,
                                     "method": meth[0], "n_ref": str(meth[2]), "rep": -1 if meth[3] is None else meth[3], "alpha": meth[4],
                                     "UAR": r["UAR"], "WAR": r["WAR"],
                                     **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])},
                                     **{f"frac_{c}": a for c, a in zip(classes[ds_name], r["frac"])}})
    print(m, "xong")
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp014_inductive_center-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

CLIP gốc xong
Paper xong
EXP-000 xong
EXP-004 e30 xong
EXP-010 e5 xong
ens chéo 004e30+010e5 xong
1824 dòng → /content/drive/MyDrive/ExpCLIP_output/exp014


# Kiểm tra pipeline
Từng seed, P0, `none` phải khớp số `test.py` (EXP-000, EXP-004); `center-test` của CLIP gốc phải khớp EXP-013 (62.11 / 46.89 / 40.62 / 52.10).

In [18]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004 e30|1|RAFDB": [58.73, 65.94], "EXP-004 e30|1|AffectNet7": [45.83, 45.83], "EXP-004 e30|1|AffectNet8": [39.6, 39.6], "EXP-004 e30|1|FERPlus": [46.14, 52.03], "EXP-004 e30|2|RAFDB": [57.07, 64.77], "EXP-004 e30|2|AffectNet7": [43.97, 43.97], "EXP-004 e30|2|AffectNet8": [38.75, 38.75], "EXP-004 e30|2|FERPlus": [45.63, 52.45], "EXP-004 e30|3|RAFDB": [58.26, 65.06], "EXP-004 e30|3|AffectNet7": [45.46, 45.46], "EXP-004 e30|3|AffectNet8": [39.07, 39.08], "EXP-004 e30|3|FERPlus": [45.34, 51.44]}
main = (res.config == "P0") & (res.prompt == BASE_PROMPT)
chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[main & (res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.method == "none")]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))
r = res[main & (res.model == "CLIP gốc") & (res.method == "center-test")].set_index("dataset").UAR[DATASETS]
print("CLIP gốc center-test:", r.round(2).to_dict(), "(EXP-013: 62.11 / 46.89 / 40.62 / 52.10)")

24/24 cấu hình so được | max |ΔUAR| = 0.42, max |ΔWAR| = 0.14
                     key  dUAR  dWAR
       EXP-000|2|FERPlus  0.42  0.02
   EXP-004 e30|1|FERPlus  0.17  0.11
         EXP-000|2|RAFDB  0.13  0.06
     EXP-004 e30|2|RAFDB -0.11 -0.14
EXP-004 e30|2|AffectNet7  0.09  0.09
   EXP-004 e30|3|FERPlus -0.06 -0.08
CLIP gốc center-test: {'RAFDB': 62.11, 'AffectNet7': 46.89, 'AffectNet8': 40.62, 'FERPlus': 52.1} (EXP-013: 62.11 / 46.89 / 40.62 / 52.10)


# Bảng 1 — So sánh cách sửa (α = 1, tham chiếu toàn bộ)

In [19]:
pd.set_option("display.width", 250)
COLS = DATASETS + ["Mean4"]
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def pv(metric="UAR"):
    t = res.pivot_table(index=["model", "seed", "prompt", "config", "method", "n_ref", "rep", "alpha"],
                        columns="dataset", values=metric)[DATASETS]
    t["Mean4"] = t[DATASETS].mean(1)
    return t.reset_index()
U, Wt = pv("UAR"), pv("WAR")

def get(m, method="none", cfg="P0", pn=BASE_PROMPT, alpha=None, n_ref=None, rep=0, t=None):
    t = U if t is None else t
    if alpha is None:
        alpha = 0.0 if method == "none" else 1.0
    if n_ref is None:
        n_ref = "None" if method in ("none", "center-test") else "all"
    q = t[(t.model == m) & (t.method == method) & (t.config == cfg) & (t.prompt == pn) & (t.alpha == alpha) & (t.n_ref == n_ref)]
    if n_ref not in ("None", "all"):
        q = q[q.rep == rep]
    return q.set_index("seed")[COLS]

MAIN_METHODS = ["none", "center-test"] + [f"center-{r}" for r in ref]
for cfg in CFG_NAMES:
    for metric, t in [("UAR", U), ("WAR", Wt)]:
        tab = {f"{m} · {me}": get(m, me, cfg, t=t).agg(fmt) for m in MODELS for me in MAIN_METHODS}
        print(f"--- {metric}, {cfg}, {BASE_PROMPT}\n{pd.DataFrame(tab).T[COLS].to_string()}\n")

--- UAR, P0, Expression Ensemble Five
dataset                                       RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
CLIP gốc · none                               52.10         40.69         36.38         38.99         42.04
CLIP gốc · center-test                        62.11         46.89         40.62         52.10         50.43
CLIP gốc · center-CAER                        58.62         46.40         40.10         43.18         47.08
CLIP gốc · center-AffNet                      61.51         45.63         40.40         47.60         48.78
Paper · none                                  57.95         44.23         38.98         43.14         46.07
Paper · center-test                           61.32         44.89         38.75         48.21         48.29
Paper · center-CAER                           57.60         44.66         38.88         39.14         45.07
Paper · center-AffNet                         61.21         43.97         39.15         45.28     

# Bảng 2 — Tiêu chí (Ensemble Five, P0, CAER-S toàn bộ, α = 1)

In [20]:
m4 = lambda m, me: get(m, me)["Mean4"]
clip_c, e10_c = m4("CLIP gốc", "center-CAER").mean(), m4("EXP-010 e5", "center-CAER")
print(f"CHÍNH: CLIP gốc + center-CAER = {clip_c:.2f} (ngưỡng 47.56 = EXP-010 e5 none {m4('EXP-010 e5', 'none').mean():.2f}) → "
      f"{'ĐẠT' if clip_c >= 47.56 else 'KHÔNG đạt'}")
for m in ["CLIP gốc", "EXP-010 e5"]:
    d = (m4(m, "center-CAER") - m4(m, "center-test"))
    print(f"PHỤ 1: {m}: center-CAER − center-test = {fmt(d)} → {'đạt' if abs(d.mean()) <= 1.0 else 'không đạt'}")
d = e10_c - clip_c
print(f"PHỤ 2: EXP-010 e5 + center-CAER − CLIP gốc + center-CAER = {fmt(d)} → {'đạt' if d.mean() >= 1.0 else 'không đạt'}")
for m in MODELS:
    print(f"  {m}: Δ center-CAER − none = {fmt(m4(m, 'center-CAER') - m4(m, 'none'))}")

CHÍNH: CLIP gốc + center-CAER = 47.08 (ngưỡng 47.56 = EXP-010 e5 none 47.56) → KHÔNG đạt
PHỤ 1: CLIP gốc: center-CAER − center-test = -3.35 → không đạt
PHỤ 1: EXP-010 e5: center-CAER − center-test = -4.58 ± 0.81 → không đạt
PHỤ 2: EXP-010 e5 + center-CAER − CLIP gốc + center-CAER = -2.85 ± 0.90 → không đạt
  CLIP gốc: Δ center-CAER − none = 5.04
  Paper: Δ center-CAER − none = -1.01
  EXP-000: Δ center-CAER − none = -0.45 ± 0.39
  EXP-004 e30: Δ center-CAER − none = -2.57 ± 0.51
  EXP-010 e5: Δ center-CAER − none = -3.33 ± 0.82
  ens chéo 004e30+010e5: Δ center-CAER − none = -3.37


# Bảng 3 — Ablation α và kích thước tập tham chiếu (P0, Ensemble Five, Mean4 UAR)

In [21]:
tab = {}
for m in MODELS:
    row = {f"α={a}": fmt(get(m, "center-CAER", alpha=a)["Mean4"]) for a in ALPHAS}
    if "AffNet" in ref:
        row |= {f"AffNet α={a}": fmt(get(m, "center-AffNet", alpha=a)["Mean4"]) for a in ALPHAS}
    for n in [5000, 500]:
        v = pd.concat([get(m, "center-CAER", n_ref=str(n), rep=r)["Mean4"] for r in range(3)])
        row[f"CAER n={n} (3 lần)"] = f"{v.mean():.2f} (min {v.min():.2f}, max {v.max():.2f})"
    tab[m] = row
print(pd.DataFrame(tab).T.to_string())

                              α=0.5         α=1.0         α=1.5  AffNet α=0.5  AffNet α=1.0  AffNet α=1.5           CAER n=5000 (3 lần)            CAER n=500 (3 lần)
CLIP gốc                      47.04         47.08         38.65         45.50         48.78         48.20  47.06 (min 46.86, max 47.19)  47.23 (min 47.00, max 47.41)
Paper                         45.62         45.07         43.54         47.19         47.40         47.01  45.06 (min 44.99, max 45.15)  45.09 (min 44.70, max 45.37)
EXP-000                45.39 ± 0.75  45.25 ± 0.78  44.87 ± 0.98  46.42 ± 0.25  47.02 ± 0.31  46.68 ± 0.17  45.30 (min 44.42, max 46.15)  45.35 (min 44.13, max 46.27)
EXP-004 e30            46.36 ± 0.87  44.42 ± 0.57  41.63 ± 0.76  48.16 ± 0.54  48.34 ± 0.24  47.80 ± 0.57  44.39 (min 43.61, max 45.02)  44.40 (min 43.37, max 45.21)
EXP-010 e5             47.47 ± 0.83  44.23 ± 0.90  39.22 ± 2.06  47.64 ± 0.34  47.68 ± 0.36  47.77 ± 0.42  44.26 (min 43.14, max 44.97)  44.35 (min 42.77, max 45.38)
ens 

# Bảng 4 — Độ lệch theo lớp: tham chiếu CAER-S so với tập test (CLIP gốc, P0)
Nếu độ lệch gần như cố định theo lớp thì b_CAER ≈ b_test (tương quan cao).

In [22]:
for ds_name in DATASETS:
    S = class_scores(None, ds_name, "P0", BASE_PROMPT)
    bt, bc = S.mean(0), ref_bias(None, "clip", ds_name, BASE_PROMPT, "CAER", "all", 0)
    t = pd.DataFrame({"b_test": bt - bt.mean(), "b_CAER": bc - bc.mean()}, index=classes[ds_name]).T   # bỏ hằng số chung
    print(f"--- {ds_name}: tương quan = {np.corrcoef(bt, bc)[0, 1]:.3f}\n{(t * 100).round(2).to_string()}\n")

--- RAFDB: tương quan = 0.953
        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
b_test      -1.85     0.14     0.55   0.79     -0.19     0.48  0.08
b_CAER      -2.62     0.72     0.50   0.50     -0.36     0.94  0.31

--- AffectNet7: tương quan = 0.957
        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
b_test      -2.66     0.19      0.1   0.89     -0.03     0.93  0.57
b_CAER      -2.62     0.72      0.5   0.50     -0.36     0.94  0.31

--- AffectNet8: tương quan = 0.968
        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear  Contempt
b_test      -2.67     0.08     0.08   0.72     -0.18     0.76  0.40      0.80
b_CAER      -2.74     0.60     0.38   0.38     -0.48     0.81  0.19      0.85

--- FERPlus: tương quan = 0.779
        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear  Contempt
b_test      -1.71    -0.07     0.89   0.22      0.19    -0.09 -0.63      1.19
b_CAER      -2.74     0.60     0.38   0.38     -0.48     0.81  0.19

# Bảng 5 — Accuracy và tỉ lệ dự đoán từng lớp (CLIP gốc và EXP-010 e5, P0)

In [23]:
for m in ["CLIP gốc", "EXP-010 e5"]:
    for ds_name in ["RAFDB", "FERPlus", "AffectNet8"]:
        t = {}
        for me in ["none", "center-CAER", "center-test"]:
            q = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT) & (res.config == "P0")
                    & (res.method == me) & (res.alpha == (0.0 if me == "none" else 1.0)) & res.n_ref.isin(["None", "all"])]
            t[f"acc {me}"] = q[[f"acc_{c}" for c in classes[ds_name]]].mean().values
            t[f"% dự đoán {me}"] = q[[f"frac_{c}" for c in classes[ds_name]]].mean().values
        t["% thật"] = np.bincount(y[ds_name]) / len(y[ds_name]) * 100
        print(f"--- {m}, {ds_name}\n{pd.DataFrame(t, index=classes[ds_name]).T.round(1).to_string()}\n")

--- CLIP gốc, RAFDB
                       Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
acc none                    17.4     52.3     41.0   90.7      50.8     70.6  41.9
% dự đoán none               6.8     15.5     30.1   18.0       9.9     13.8   5.9
acc center-CAER             90.7     45.2     30.7   90.7      66.0     51.9  35.1
% dự đoán center-CAER       42.0      9.5      9.5   18.4      10.6      6.0   4.0
acc center-test             75.6     60.0     31.0   80.2      61.7     77.5  48.6
% dự đoán center-test       31.2     18.1     11.7    9.5      10.4     11.7   7.3
% thật                      38.6     15.6     22.2    5.3      10.7      5.2   2.4

--- CLIP gốc, FERPlus
                       Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear  Contempt
acc none                     7.9     33.6     32.9   47.5      76.0     33.3  17.3      63.3
% dự đoán none               2.1      6.9     26.8    6.0      14.9      2.4   0.7      40.4
acc center-CAE

# Lưu JSON (cùng định dạng `research_log/results/`)

In [24]:
out = {"source": "notebooks/EXP-014_inductive_center.ipynb", "refs": {k: REF_FEATS[k] for k in ref},
       "sanity_check": {"max_abs_dUAR": round(float(chk.dUAR.abs().max()), 2), "max_abs_dWAR": round(float(chk.dWAR.abs().max()), 2)},
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "method": r.method,
                 "n_ref": r.n_ref, "rep": int(r.rep), "alpha": r.alpha,
                 "dataset": r.dataset, "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-014-inductive-center.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-014-inductive-center.json")

1824 dòng → /content/drive/MyDrive/ExpCLIP_output/exp014/EXP-014-inductive-center.json
